# FaithfulMed — Extractor Vector Store
Owner: Vaibhavi. This September notebook loads Ruhma's MedlinePlus vectors into ChromaDB and retrieves three source-attributed definitions. Read PROJECT_CONTEXT.md for verified progress.

Keep the Python files next to this notebook. Install dependencies using `%pip install -r requirements.txt` in a new cell. Then edit EMBEDDINGS_PATH below. In Colab use `/content/faithfulmed_chroma` for the live DB, rather than mounted Drive. Input embeddings can remain on Drive. The first query downloads MiniLM weights; no API key is needed.

In [1]:
import os
from pathlib import Path
from rag import read_embeddings, ingest, retrieve, format_context, MODEL, COLLECTION

EMBEDDINGS_PATH = Path(os.environ.get("EMBEDDINGS_PATH", "../data/RAG_Embeddings_Full.jsonl"))
DB_PATH = os.environ.get("FAITHFULMED_DB_PATH", "./chroma_data")
if not EMBEDDINGS_PATH.is_file():
    raise FileNotFoundError("Set EMBEDDINGS_PATH to your downloaded JSONL before continuing.")
print({"model": MODEL, "collection": COLLECTION})

{'model': 'sentence-transformers/all-MiniLM-L6-v2', 'collection': 'faithfulmed_medlineplus_minilm_v1'}


## Audit the real handoff
Only MedlinePlus is selected; other sources are counted and excluded. Each selected record must include term, definition, source link, embedded text, and a finite nonzero 384-dimensional vector. The model is documented by Ruhma's updated notebook; dimension alone does not identify it.

In [2]:
records, report = read_embeddings(EMBEDDINGS_PATH)
report

{'dimension': 384,
 'duplicate_medlineplus_records': 0,
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'indexed_records': 1015,
 'sources': {'MedQuAD': 16407, 'MedlinePlus': 1015, 'PLABA': 921},
 'total_records': 18343}

## Create the persistent database
Stable IDs and upsert prevent duplication when rerunning the same handoff. Each supplied topic stays one record. Changing text creates new IDs; use a versioned collection for a revised snapshot.

In [3]:
first = ingest(EMBEDDINGS_PATH, DB_PATH)
second = ingest(EMBEDDINGS_PATH, DB_PATH)
assert first["collection_count"] == second["collection_count"] == report["indexed_records"]
print({"records": second["collection_count"], "repeat_ingestion": "same count"})

{'records': 1015, 'repeat_ingestion': 'same count'}


## Search medical terms
The query uses the same MiniLM model as the stored vectors. Distances are cosine distances, not confidence scores. These smoke examples demonstrate functionality; they do not establish evaluated accuracy.

In [4]:
search_observations = []
for query in ["hypertension", "diabetes", "myocardial infarction", "A1C", "asthma"]:
    hits = retrieve(query, DB_PATH, top_k=3)
    search_observations.append({"query": query, "top_3": [r["metadata"]["term"] for r in hits]})
search_observations

[{'query': 'hypertension',
  'top_3': ['Blood Pressure Medicines',
            'How to Prevent High Blood Pressure',
            'High Blood Pressure']},
 {'query': 'diabetes',
  'top_3': ['Diabetes', 'Diabetic Foot', 'Diabetes Medicines']},
 {'query': 'myocardial infarction',
  'top_3': ['Cardiomyopathy', 'Heart Attack', 'Heart Disease in Women']},
 {'query': 'A1C', 'top_3': ['A1C', 'Blood', 'Vitamins']},
 {'query': 'asthma',
  'top_3': ['Asthma', 'Asthma in Children', 'Breathing Problems']}]

## Simplifier / Verifier handoff
`retrieve()` returns JSON-serializable records with ID, text, clean definition, metadata, and distance. `format_context(hits)` creates numbered evidence with source links.

For Rafi's existing repository adapter, pass `TextQueryCollection(DB_PATH)` into `retrieve_context(..., top_k=3)`. This wrapper uses matching MiniLM query vectors instead of Chroma's default embedder.

MTSamples and Synthea are clinical input datasets; MedlinePlus is the supporting lay-language knowledge base. Their EDA is documented in MTSAMPLES_EDA.md and SYNTHEA_EDA.md. The FHIR helper flattens selected resource fields; it is not yet the October LLM clinical-atom extractor.

Limitations: MiniLM's 256-token limit truncates long topics; lower-ranked matches can be weak; no frozen-set retrieval evaluation is complete; the official Gemini embedding milestone needs a coordinated model migration. Update PROJECT_CONTEXT.md after meaningful work.